# lm-evaluation-harness로 한국어 LLM 평가하기

`lm-evaluation-harness`는 여러 언어 모델을 동일한 벤치마크와 평가 조건으로 비교하는 오픈소스 평가 프레임워크이다. 모델 backend, task, few-shot 수, seed와 출력 경로를 명령으로 고정해 재현 가능한 평가를 수행한다.

### 실행 환경

- RunPod NVIDIA GPU, Python 3.12, PyTorch 2.8을 기준으로 하며 GPU VRAM은 24GB 이상을 권장한다.
- `HF_TOKEN`, Llama 3.1 모델 접근 승인과 모델·데이터셋 다운로드가 가능한 네트워크가 필요하다.
- 로컬 CPU 실행은 권장하지 않으며 token 문자열은 노트북에 기록하거나 출력하지 않는다.

### 실습 흐름

`고정 버전 설치 → task YAML → 모델·task 실행 → 10건 결과 집계 → sample 오류 분석`

![lm-evaluation-harness task 실행 흐름](https://cdn.jsdelivr.net/gh/goat-skn-ai/image-repo@bbdd290451cb2e955a3e1ae4e9f76e795947aa27/08_llm/14_lm_evaluation/02_lm_eval_harness/lm_eval_task_pipeline.svg)

모델 backend는 Llama checkpoint를 불러오고, task YAML은 데이터 한 행을 prompt·정답·metric으로 변환한다. 결과는 집계 점수와 문항별 sample 로그로 나뉘며 두 결과를 함께 읽어야 점수 원인을 설명할 수 있다.


## 패키지 설치와 Kernel 재시작

PyTorch 2.8은 RunPod image에 준비된 GPU build를 그대로 사용하고 이 셀에서 다시 설치하지 않는다. 평가 프레임워크와 결과 표에 직접 필요한 여섯 패키지를 실습 기준 버전으로 고정한다. `lm-eval`은 core만 설치하고 HF backend 의존성을 직접 명시해 직전 실습의 `peft==0.18.0`이 바뀌지 않게 한다.

설치 셀을 한 번 실행한 뒤 **PyCharm의 Jupyter kernel을 한 번 재시작**한다. 재시작하지 않으면 설치 전 모듈이 메모리에 남아 import 오류나 버전 불일치가 생길 수 있다. 재시작 후에는 다음 `버전 확인과 Hugging Face 로그인` 셀부터 실행한다.

In [ ]:
import os

# tokenizer를 import하기 전에 설정해 불필요한 병렬 처리 경고를 줄인다.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# RunPod의 PyTorch 2.8은 유지하고 harness와 직접 연동되는 패키지만 설치한다.
%pip install "lm-eval==0.4.12" "transformers==4.56.2" "accelerate==1.14.0" "huggingface_hub==0.36.2" "datasets==4.8.5" "pandas==2.3.3"


## 버전 확인과 Hugging Face 로그인

kernel 재시작 후 Python·PyTorch·평가 패키지 버전을 확인한다. `login()`은 RunPod 환경 변수의 `HF_TOKEN`을 Hugging Face client의 인증 정보로 연결한다. token에 `meta-llama/Meta-Llama-3.1-8B` 접근 승인이 있어야 모델 다운로드가 성공한다.

In [ ]:
# 설치된 distribution 버전과 실제 GPU runtime을 한 화면에서 비교한다.
from importlib.metadata import version
import os
import platform

import accelerate
import datasets
import huggingface_hub
import lm_eval
import pandas as pd
import torch
import transformers
from huggingface_hub import login

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("lm-eval:", version("lm-eval"))
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "사용 불가")

# token 원문은 출력하지 않고 환경 변수에서만 읽는다.
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("RunPod 환경 변수 HF_TOKEN을 확인한다.")
login(token=hf_token, add_to_git_credential=False)


## 설치된 task YAML 확인

PyPI로 설치한 `lm-eval` 패키지 안에도 공식 task YAML이 포함된다. GitHub `main`을 따로 복제하지 않고 `lm_eval.__file__`을 기준으로 현재 실행 중인 0.4.12 패키지의 task 경로를 찾는다. 이렇게 해야 CLI 코드와 읽어 보는 YAML의 버전이 일치한다.

KoBEST BoolQ의 `output_type: multiple_choice`는 `아니오`, `예` 두 선택지의 로그 가능도를 비교한다. `doc_to_text`, `doc_to_choice`, `metric_list`가 각각 prompt, 후보와 집계 지표를 정의한다.

In [ ]:
from pathlib import Path

# lm_eval.__file__의 부모가 현재 설치 패키지이고 그 아래 tasks에 공식 YAML이 있다.
# clone 경로가 아니라 import된 package 경로를 사용해 CLI와 YAML 버전을 맞춘다.
TASK_ROOT = Path(lm_eval.__file__).resolve().parent / "tasks"
KOBEST_BOOLQ_YAML = TASK_ROOT / "kobest" / "kobest_boolq.yaml"

print("task root:", TASK_ROOT)
print(KOBEST_BOOLQ_YAML.read_text(encoding="utf-8"))


## KoBEST BoolQ 10건 zero-shot 평가

`--limit 10`은 실습에서 모델 다운로드와 평가 pipeline이 동작하는지 확인하기 위한 기능 점검용이다. 10건 점수는 표본이 너무 작으므로 **정식 벤치마크 점수나 모델 우열의 근거로 사용하지 않는다**.

`--batch_size 1`은 처리량보다 GPU 메모리 안정성을 우선한다. `--log_samples`는 집계 결과와 함께 문항별 입력·응답을 JSONL로 남긴다.

In [ ]:
%%bash
lm-eval run --model hf \
    --model_args pretrained=meta-llama/Meta-Llama-3.1-8B,dtype=auto \
    --tasks kobest_boolq \
    --num_fewshot 0 \
    --limit 10 \
    --seed 42 \
    --device cuda:0 \
    --batch_size 1 \
    --output_path ./eval_llama31-8B/kobest/boolq/0-shot \
    --log_samples \
    --show_config


### 집계 JSON을 간단한 표로 읽기

CLI의 출력 폴더 구조에는 모델 이름과 실행 시간이 포함될 수 있다. 따라서 고정 파일명을 가정하지 않고 최신 `results_*.json`을 찾아 task별 수치 지표만 DataFrame으로 정리한다.

In [ ]:
import json
import pandas as pd

# 실행 시각별 하위 폴더를 포함하므로 root 아래를 재귀 검색한다.
KOBEST_RESULT_ROOT = Path("./eval_llama31-8B/kobest/boolq/0-shot")
result_paths = list(KOBEST_RESULT_ROOT.rglob("results_*.json"))
if not result_paths:
    raise FileNotFoundError("먼저 KoBEST 평가 셀을 실행한다.")

# 재실행한 경우 수정 시간이 가장 최근인 결과 JSON을 선택한다.
result_path = max(result_paths, key=lambda path: path.stat().st_mtime)
result_payload = json.loads(result_path.read_text(encoding="utf-8"))

# task 이름과 표준오차가 아닌 수치 metric만 한 행 딕셔너리로 변환한다.
rows = [
    {
        "task": task_name,
        **{
            metric_name: value
            for metric_name, value in metrics.items()
            if isinstance(value, (int, float)) and not metric_name.endswith("_stderr")
        },
    }
    for task_name, metrics in result_payload["results"].items()
]

result_df = pd.DataFrame(rows)
display(result_df)


### sample JSONL로 문항별 결과 읽기

sample 로그의 `doc`은 원문 문항, `target`은 정답, `resps`는 모델 backend의 원응답, `filtered_resps`는 metric에 전달된 후처리 결과이다. 저장된 column 중 핵심 column만 골라 앞의 세 문항을 확인한다.

In [ ]:
# 집계 점수와 같은 실행에서 저장된 문항별 JSONL을 재귀 검색한다.
sample_paths = list(KOBEST_RESULT_ROOT.rglob("samples_kobest_boolq_*.jsonl"))
if not sample_paths:
    raise FileNotFoundError("KoBEST 평가 결과의 sample JSONL을 확인한다.")

sample_path = max(sample_paths, key=lambda path: path.stat().st_mtime)
samples_df = pd.read_json(sample_path, lines=True)
# lm-eval 버전에 따라 없는 column은 제외해 핵심 정보만 안전하게 표시한다.
sample_columns = [
    column
    for column in ["doc", "target", "resps", "filtered_resps"]
    if column in samples_df.columns
]
display(samples_df[sample_columns].head(3))


## [선택] KMMLU 생성 평가

`kmmlu_direct_food_processing`은 A~D 중 답을 직접 생성하고 exact match로 채점한다. KoBEST의 선택지 로그 가능도 방식과 달리 지식뿐 아니라 출력 형식도 점수에 영향을 준다.

In [ ]:
KMMLU_DIRECT_YAML = TASK_ROOT / "kmmlu" / "direct" / "_direct_kmmlu_yaml"
print(KMMLU_DIRECT_YAML.read_text(encoding="utf-8"))


### [선택] KMMLU 10건 실행

KoBEST와 모델·few-shot·seed·device 조건을 유지하고 task만 KMMLU로 바꾼다. `--limit 10`과 `--batch_size 1`은 기능 확인과 GPU 안정성을 위한 설정이며 결과를 정식 KMMLU 점수로 해석하지 않는다.

In [ ]:
%%bash
lm-eval run --model hf \
    --model_args pretrained=meta-llama/Meta-Llama-3.1-8B,dtype=auto \
    --tasks kmmlu_direct_food_processing \
    --num_fewshot 0 \
    --limit 10 \
    --seed 42 \
    --device cuda:0 \
    --batch_size 1 \
    --output_path ./eval_llama31-8B/kmmlu/direct_food_processing/0-shot \
    --log_samples \
    --show_config


### [선택] KMMLU sample 확인

집계 exact match만으로는 지식 오류와 출력 형식 오류를 구분할 수 없다. `target`, `resps`, `filtered_resps`를 나란히 읽어 모델이 정답 문자를 몰랐는지, 부가 설명 때문에 채점에서 실패했는지 확인한다.

In [ ]:
# 선택 평가를 실행한 output root 아래에서 최신 sample 파일을 찾는다.
KMMLU_RESULT_ROOT = Path("./eval_llama31-8B/kmmlu/direct_food_processing/0-shot")
kmmlu_sample_paths = list(KMMLU_RESULT_ROOT.rglob("samples_kmmlu_direct_food_processing_*.jsonl"))
if not kmmlu_sample_paths:
    raise FileNotFoundError("먼저 선택 KMMLU 평가 셀을 실행해야 한다.")

kmmlu_sample_path = max(kmmlu_sample_paths, key=lambda path: path.stat().st_mtime)
kmmlu_samples_df = pd.read_json(kmmlu_sample_path, lines=True)
# target과 모델 응답을 나란히 볼 수 있는 column만 선택한다.
kmmlu_columns = [
    column
    for column in ["doc", "target", "resps", "filtered_resps"]
    if column in kmmlu_samples_df.columns
]
display(kmmlu_samples_df[kmmlu_columns].head(5))


## [선택] 정답 문자 추출 필터

기본 task 파일을 수정하지 않고 새 YAML을 `--include_path`로 추가한다. 정규식은 출력 맨 앞의 공백과 선택적인 `정답:` 또는 `answer:` 접두어를 허용한 뒤 첫 A~D 문자를 추출한다.

필터는 답변 형식을 정리할 뿐 모델의 지식과 추론을 개선하지 않는다. 같은 모델 출력도 추출 규칙이 달라지면 점수가 변할 수 있으므로 필터 설정을 평가 결과와 함께 보관한다.

In [ ]:
# 기본 task를 덮어쓰지 않고 새 이름과 후처리 규칙을 한 YAML로 정의한다.
CUSTOM_TASK_YAML = r''' 
task: kmmlu_direct_food_processing_filtered
dataset_path: HAERAE-HUB/KMMLU
dataset_name: Food-Processing
output_type: generate_until
test_split: test
fewshot_split: dev
doc_to_text: "{{question.strip()}}\nA. {{A}}\nB. {{B}}\nC. {{C}}\nD. {{D}}\n정답："
doc_to_target: "{{['A', 'B', 'C', 'D'][answer-1]}}"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_case: true
    ignore_punctuation: true
    regexes_to_ignore:
      - " "
generation_kwargs:
  until:
    - "Q:"
    - "\n\n"
    - "</s>"
    - "."
  do_sample: false
  temperature: 0.0
filter_list:
  - name: get-answer
    filter:
      - function: regex
        regex_pattern: '(?i)^\s*(?:(?:정답|answer)\s*[:：]?\s*)?([A-D])\b'
      - function: take_first
metadata:
  version: 2.0
'''

# include_path가 읽을 별도 폴더에 UTF-8 YAML을 저장한다.
TEMPLATE_DIR = Path("./template")
CUSTOM_YAML_PATH = TEMPLATE_DIR / "kmmlu_direct_food_processing_filtered.yaml"
TEMPLATE_DIR.mkdir(exist_ok=True)
CUSTOM_YAML_PATH.write_text(CUSTOM_TASK_YAML.strip() + "\n", encoding="utf-8")
print(CUSTOM_YAML_PATH)


### [선택] 필터 task 10건 재평가

`--include_path ./template`으로 방금 만든 YAML을 task 검색 경로에 추가한다. 모델·표본·seed는 앞 평가와 같게 유지하고 추출 규칙만 바꾸어 점수 변화가 후처리에서 비롯되었는지 비교한다.

In [ ]:
%%bash
lm-eval run --model hf \
    --model_args pretrained=meta-llama/Meta-Llama-3.1-8B,dtype=auto \
    --include_path ./template \
    --tasks kmmlu_direct_food_processing_filtered \
    --num_fewshot 0 \
    --limit 10 \
    --seed 42 \
    --device cuda:0 \
    --batch_size 1 \
    --output_path ./eval_llama31-8B/kmmlu/direct_food_processing_filtered/0-shot \
    --log_samples \
    --show_config


### [선택] 필터 전후 sample 확인

원응답 `resps`와 필터 결과 `filtered_resps`를 target 옆에 놓고 비교한다. 정답 문자가 같아졌다면 형식 보정 효과이고, 추출 후에도 target이 다르면 모델 자체의 오답이다.

In [ ]:
# 커스텀 task의 output root에서 최신 필터 적용 sample을 찾는다.
FILTERED_RESULT_ROOT = Path("./eval_llama31-8B/kmmlu/direct_food_processing_filtered/0-shot")
filtered_paths = list(FILTERED_RESULT_ROOT.rglob("samples_kmmlu_direct_food_processing_filtered_*.jsonl"))
if not filtered_paths:
    raise FileNotFoundError("먼저 선택 필터 평가 셀을 실행한다.")

filtered_path = max(filtered_paths, key=lambda path: path.stat().st_mtime)
filtered_df = pd.read_json(filtered_path, lines=True)
# 정답, 원응답과 필터 출력만 골라 변환 효과를 직접 비교한다.
filtered_columns = [
    column
    for column in ["target", "resps", "filtered_resps"]
    if column in filtered_df.columns
]
display(filtered_df[filtered_columns].head(5))


## 정리

KoBEST 10건 데모의 목적은 `고정 버전 패키지 → task YAML → 모델 backend → 집계 결과 → sample 로그` 연결을 확인하는 것이다. 정식 모델 비교에서는 `--limit`을 제거하고 전체 test split, task 버전, 모델 revision, few-shot, seed, batch와 필터 설정을 함께 기록한다.

KMMLU 선택 실습은 같은 모델도 로그 가능도 채점, 직접 생성과 정규식 후처리 같은 평가 계약에 따라 점수가 달라질 수 있음을 보여 준다. 따라서 최종 보고서에는 평균 점수뿐 아니라 sample 오류 유형도 함께 남긴다.